# 01 数据清洗与时间规律分析（成员A）

题目四：城市电力负荷分析与预测

本 notebook 负责：数据读入、列名规整、时间解析、缺失/重复/网格完整性检查，以及日、周、季节尺度的负荷规律分析。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Windows 下显示中文与负号
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print('pandas', pd.__version__)
print('numpy', np.__version__)
print('matplotlib', plt.matplotlib.__version__)

pandas 3.0.5
numpy 2.5.2
matplotlib 3.11.0


## 1. 读入数据与字段规整

原始列名中 `Zone 2  Power Consumption` 含双空格，统一改成短名：

| 短名 | 原列名 |
|---|---|
| DateTime | DateTime |
| T | Temperature |
| RH | Humidity |
| WS | Wind Speed |
| GDF | general diffuse flows |
| DF | diffuse flows |
| Z1 | Zone 1 Power Consumption |
| Z2 | Zone 2  Power Consumption |
| Z3 | Zone 3  Power Consumption |

时间列格式为 **月/日/年 时:分**，解析时必须指定 `format='%m/%d/%Y %H:%M'`，否则会出现日期错位。

In [2]:
df = pd.read_csv('data/题4_城市电力负荷.csv', encoding='utf-8-sig')

# 统一列名（按位置重命名，避免原列名的双空格问题）
df.columns = ['DateTime', 'T', 'RH', 'WS', 'GDF', 'DF', 'Z1', 'Z2', 'Z3']

# 解析时间并设为索引
df['DateTime'] = pd.to_datetime(df['DateTime'], format='%m/%d/%Y %H:%M')
df = df.set_index('DateTime').sort_index()

print('shape:', df.shape)
print('时间范围:', df.index.min(), '->', df.index.max())
print('缺失值合计:', df.isna().sum().sum())
print('重复时间戳:', df.index.duplicated().sum())
df.head()

shape: (52416, 8)
时间范围: 2017-01-01 00:00:00 -> 2017-12-30 23:50:00
缺失值合计: 0
重复时间戳: 0


,T,RH,WS,GDF,DF,Z1,Z2,Z3
DateTime,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373
2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711
2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964


## 2. 时间网格完整性检查

预期：2017-01-01 00:00 至 2017-12-30 23:50，每 10 分钟一个点，共 364 × 144 = 52,416 行，无缺失、无重复。

In [3]:
full_grid = pd.date_range(df.index.min(), df.index.max(), freq='10min')

print('应有时间点:', len(full_grid))
print('实际行数:', len(df))
print('缺失时间点:', len(full_grid.difference(df.index)))
print('覆盖天数:', df.index.normalize().nunique())

应有时间点: 52416
实际行数: 52416
缺失时间点: 0
覆盖天数: 364


## 3. 数值字段概况

下一步：用 `describe()` 和箱线图检查异常值，重点记录风速（WS）的双峰分布现象。

In [4]:
df.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
T,52416.0,18.810,5.815,3.247,14.410,18.780,22.890,40.010
RH,52416.0,68.260,15.551,11.340,58.310,69.860,81.400,94.800
WS,52416.0,1.959,2.349,0.050,0.078,0.086,4.915,6.483
GDF,52416.0,182.697,264.401,0.004,0.062,5.036,319.600,1163.000
DF,52416.0,75.028,124.211,0.011,0.122,4.456,101.000,936.000
Z1,52416.0,32344.971,7130.563,13895.696,26310.669,32265.920,37309.018,52204.395
Z2,52416.0,21042.509,5201.466,8560.081,16980.766,20823.168,24713.718,37408.861
Z3,52416.0,17835.406,6622.165,5935.174,13129.327,16415.117,21624.100,47598.326
